# 데이터 전처리(Data Preprocessing) 정리 노트

결측치 처리 → X/y 분리 → 인코딩 → 피처 스케일링 → Train/Test 분리까지의 전체 파이프라인을 정리한 노트북입니다.

> 이 노트북은 `X = ct.fit_transform(X)`처럼 **자기 자신을 덮어쓰는 코드**가 셀을 재실행할 때마다 컬럼이 계속 늘어나는 버그를 피하기 위해, 단계마다 새로운 변수명(`X_encoded`, `X_scaled` 등)을 사용합니다. 이렇게 하면 어떤 셀을 몇 번 재실행해도 항상 같은 결과가 나와요.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. 데이터 불러오기

(원본 CSV 대신, 실습에서 사용한 데이터를 그대로 재현했습니다. 실제 파일을 쓰신다면 `pd.read_csv(...)`로 바꿔주세요.)

In [ ]:
df = pd.DataFrame({
    'Country':  ['France', 'Spain', 'Germany', 'Spain', 'Germany',
                 'France', 'Spain', 'France', 'Germany', 'France'],
    'Age':      [44.0, 27.0, 30.0, 38.0, 40.0, 35.0, np.nan, 48.0, 50.0, 37.0],
    'Salary':   [72000.0, 48000.0, 54000.0, 61000.0, np.nan,
                 58000.0, 52000.0, 79000.0, 83000.0, 67000.0],
    'Purchased': ['No', 'Yes', 'No', 'No', 'Yes',
                  'Yes', 'No', 'Yes', 'No', 'Yes']
})
df

## 2. 결측치(Missing Value) 확인

`Age`, `Salary`에 각각 1개씩 결측치가 있습니다.

In [ ]:
df.isnull().sum()

## 3. 결측치 처리 전략

- **0으로 채우기 (`fillna(0)`)**: 비추천. Age=0, Salary=0은 실제로 있을 수 없는 값이라 평균/분포 자체가 왜곡됩니다.
- **행 삭제 (`dropna()`)**: 데이터가 10행뿐이라 20%를 버리는 셈 → 지금처럼 데이터가 적을 땐 비추천.
- **평균(mean)으로 채우기**: 표준적인 방법. 이상치가 있다면 median도 고려. → **이 노트북에서 사용**

세 방법을 비교만 해보고, 실제로는 평균 대체를 df에 적용합니다.

In [ ]:
# 비교용 (원본 df는 건드리지 않음)
df_zero = df.fillna({'Age': 0, 'Salary': 0})
df_mean = df.fillna({'Age': df['Age'].mean(), 'Salary': df['Salary'].mean()})

print('0으로 채운 경우 Age 평균:', df_zero['Age'].mean().round(2))
print('평균으로 채운 경우 Age 평균:', df_mean['Age'].mean().round(2))

In [ ]:
# 실제 적용: 평균 대체
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Salary'] = df['Salary'].fillna(df['Salary'].mean())
df

## 4. X, y 분리 (독립변수 / 종속변수)

`Purchased`가 예측 대상(y), 나머지가 독립변수(X)입니다.

In [ ]:
X = df.drop('Purchased', axis=1)
y = df['Purchased']
X

## 5. 인코딩 (Encoding)

| 대상 | 방법 | 이유 |
|---|---|---|
| `Country` (X, 3개 범주, 순서 없음) | One-Hot Encoding | 순서가 없는데 Label Encoding을 쓰면 모델이 없는 서열 정보를 학습해버림 |
| `Purchased` (y, 이진값) | Label Encoding | 값이 2개뿐이라 서열 문제가 생기지 않음 |

**주의**: `ColumnTransformer.fit_transform(X)` 결과를 다시 `X`에 덮어쓰면, 이 셀을 재실행할 때마다 이미 인코딩된 0/1 컬럼을 또 인코딩해버려서 컬럼 수가 계속 불어나는 버그가 생깁니다. 아래처럼 **새 변수명(`X_encoded`)**을 쓰면 몇 번을 재실행해도 항상 같은 결과가 나와서 안전합니다.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

ct = ColumnTransformer(
    [('encoder', OneHotEncoder(), [0])],  # 0번 컬럼 = Country
    remainder='passthrough'               # 나머지(Age, Salary)는 그대로 유지
)

X_encoded = ct.fit_transform(X)  # X 자체는 그대로 두고 결과만 새 변수에 저장
print(ct.get_feature_names_out())
X_encoded

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)  # No=0, Yes=1
y_encoded

## 6. Feature Scaling

`Country`는 이미 0/1이라 스케일링이 필요 없고, `Age`(20~50대)와 `Salary`(4~9만)는 단위 차이가 커서 스케일링이 필요합니다. `ColumnTransformer` 출력 순서는 [France, Germany, Spain, Age, Salary]이므로, 뒤쪽 2개 컬럼(인덱스 3번부터)만 스케일링합니다.

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = X_encoded.copy()
scaler = StandardScaler()
X_scaled[:, 3:] = scaler.fit_transform(X_scaled[:, 3:])
X_scaled

## 7. Train / Test 분리

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    random_state=10
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X_train, X_test, y_train, y_test

## 정리

| 단계 | 핵심 함수 | 기억할 점 |
|---|---|---|
| 결측치 처리 | `fillna(mean())` | 0으로 채우면 분포가 왜곡됨 |
| X/y 분리 | `df.drop()`, `df[...]` | Purchased가 y |
| 인코딩 | `OneHotEncoder` + `ColumnTransformer` | `remainder='passthrough'` 빠뜨리면 나머지 컬럼이 사라짐 |
| y 인코딩 | `LabelEncoder` | 이진값이라 서열 문제 없음 |
| 스케일링 | `StandardScaler` | 원핫 인코딩된 0/1 컬럼은 스케일링 대상에서 제외 |
| 분리 | `train_test_split` | `random_state` 고정하면 재현 가능 |

**재실행 안전 습관**: `X = 어떤함수(X)` 형태로 자기 자신을 덮어쓰는 코드는 셀을 두 번 실행하면 결과가 달라질 수 있어요. 이 노트북처럼 단계마다 `X_encoded`, `X_scaled`처럼 새 변수명을 쓰면 어떤 셀을 몇 번 재실행해도 항상 같은 결과가 나옵니다. 그래도 결과가 이상하면 **Restart Kernel → Run All**로 처음부터 다시 실행해서 상태를 초기화하세요.